In [4]:
from langchain_groq import ChatGroq
from dotenv  import load_dotenv
from langgraph.graph import StateGraph, START, END, MessagesState
from langchain_core.messages import HumanMessage,SystemMessage
from pydantic import BaseModel,Field
from typing import Optional


In [2]:
load_dotenv()
llm = ChatGroq(model="openai/gpt-oss-120b")


In [5]:
class LeadState(MessagesState):
    user_name:str
    project_type:Optional[str]
    budget:Optional[str]
    is_complete:bool

In [7]:
def ExtarctedLeadInfo(BaseModel):
    user_name:Optional[str]=Field(default=None,description="The name of the user if mentioned, otherwise null")
    project_type: Optional[str] = Field(default=None,description="The type of software or project the user wants to build (e.g., website, mobile app, AI bot), otherwise null.")
    budget: Optional[str] = Field(default=None,description="The budget mentioned by the user (e.g., $500, 50k INR), otherwise null.")

extractor_llm=llm.with_structured_output(ExtarctedLeadInfo)

In [9]:
def info_extractor_node(state:LeadState):
    latest_user_msg=state["messages"][-1].content
    extraction_prompt=[SystemMessage(
        content=("Extract the user's name, project type, and budget from the message below. "
            "Only extract values that are explicitly stated. If a value is not mentioned, return null for that field.")
            ),
        HumanMessage(
            content=latest_user_msg
        )
    ]
    extracted_data:ExtarctedLeadInfo=extractor_llm.invoke(extraction_prompt)

    updated_name=extracted_data.user_name or state.get("user_name")
    updated_project=extracted_data.project_type or state.get("project_type")
    updated_budget = extracted_data.budget or state.get("budget")

    all_filled=bool(updated_name and updated_project and updated_budget)

    return{
        "user_name": updated_name,
        "project_type": updated_project,
        "budget": updated_budget,
        "is_complete": all_filled
    }

In [12]:
def receptionist_bot_node(State:LeadState):
    name=State.get("user_name")
    project=State.get("project_type")
    budget = State.get("budget")
    is_complete = State.get("is_complete", False)

    missing_fields=[]
    if not name:
        missing_fields.append("user's name")
    if not project:
        missing_fields.append("project type (what they want to build)")
    if not budget:
        missing_fields.append("estimated budget")
    if is_complete:
        system_instructions=(
            "You are a friendly agency receptionist. All required lead details are now collected.\n"
            f"- Name: {name}\n"
            f"- Project Type: {project}\n"
            f"- Budget: {budget}\n"
            "Thank the user warmly, display this summary clearly, and tell them our senior consultant will contact them soon."
        )
    else:
        system_instructions = (
            "You are a friendly software agency receptionist collecting project inquiry details.\n"
            f"Currently collected -> Name: {name}, Project: {project}, Budget: {budget}.\n"
            f"Still missing -> {', '.join(missing_fields)}.\n"
            "Naturally ask the user for the next missing detail in 1-2 short sentences. Do not ask for everything at once."
        )
    prompt_messages=[SystemMessage(content=system_instructions) + State["messages"]]
    response=llm.invoke(prompt_messages)
    return { "messages"[response]}

In [13]:


builder=StateGraph(LeadState)

In [14]:
builder.add_node("info_extractor_node",info_extractor_node)
builder.add_node("receptionlist_bot_node",receptionist_bot_node)

builder.add_edge(START,"info_extractor_node")
builder.add_edge("info_extractor_node","receptionlist_bot_node")
builder.add_edge("receptionlist_bot_node",END)

lead_bot_graph=builder.compile()

In [ ]:
def run_lead_bot():
    print("Agency Receptionist Bot started! Type 'exit' or 'quit' to stop.\n")
    print("AI: Hello! Welcome to TechCraft Agency. How can I help you today?\n")

    current_state:LeadState={
        
    }